# Multiagent LP/MILP Solver (Colab)

This notebook builds a **multiagent framework** with at least two LLM agents to:
1. Convert a natural-language optimization problem into a linear model (LP/MILP).
2. Critique and correct the model.
3. Solve it with PuLP/CBC.
4. Provide analysis when the model is optimal or not solvable.

The two LLM agents use free/open models from Hugging Face:
- `google/flan-t5-small` (Formulator Agent)
- `google/flan-t5-base` (Critic Agent)


In [4]:
# Colab setup
# These are free/open-source libraries. No paid API is required.
!pip -q install pulp transformers sentencepiece accelerate

In [5]:
import json
import re
from dataclasses import dataclass
from typing import Dict, Any, Tuple

import pulp
from transformers import pipeline
from IPython.display import Markdown, display

In [6]:
# -----------------------------
# LLM Agent definitions
# -----------------------------

@dataclass
class HFText2TextAgent:
    name: str
    model_id: str
    max_new_tokens: int = 512
    temperature: float = 0.0

    def __post_init__(self):
        # device_map='auto' lets Colab use GPU if available.
        self.pipe = pipeline(
            "text2text-generation",
            model=self.model_id,
            tokenizer=self.model_id,
            device_map="auto"
        )

    def run(self, prompt: str) -> str:
        out = self.pipe(
            prompt,
            max_new_tokens=self.max_new_tokens,
            do_sample=self.temperature > 0,
            temperature=self.temperature,
            truncation=True
        )
        return out[0]["generated_text"].strip()


# Two separate LLM agents (free/open models)
formulator_agent = HFText2TextAgent(
    name="FormulatorAgent",
    model_id="google/flan-t5-small",
    max_new_tokens=700,
    temperature=0.0,
)

critic_agent = HFText2TextAgent(
    name="CriticAgent",
    model_id="google/flan-t5-base",
    max_new_tokens=700,
    temperature=0.0,
)

print("Agents loaded:", formulator_agent.model_id, "and", critic_agent.model_id)

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
# -----------------------------
# Parsing and schema utilities
# -----------------------------

def _extract_json_block(text: str) -> str:
    """Extract a JSON object from noisy LLM output."""
    text = text.strip()

    # Remove code fences if present.
    text = text.replace("```json", "```")
    if "```" in text:
        parts = text.split("```")
        # Pick the longest chunk as likely JSON payload.
        text = max(parts, key=len).strip()

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in LLM output.")

    return text[start:end + 1]


def parse_json_output(text: str) -> Dict[str, Any]:
    payload = _extract_json_block(text)
    return json.loads(payload)


def normalize_schema(model: Dict[str, Any]) -> Dict[str, Any]:
    """Normalize and minimally validate model schema."""
    required = ["problem_type", "objective", "variables", "constraints"]
    for key in required:
        if key not in model:
            raise ValueError(f"Missing required key: {key}")

    sense = model["objective"].get("sense", "max").lower()
    if sense not in {"max", "min"}:
        raise ValueError("Objective sense must be 'max' or 'min'.")

    # Fill defaults for variable metadata.
    for v, meta in model["variables"].items():
        meta.setdefault("type", "continuous")
        meta.setdefault("low", 0)
        meta.setdefault("up", None)

    # Ensure each constraint has expected keys.
    for c in model["constraints"]:
        for k in ["lhs", "sense", "rhs"]:
            if k not in c:
                raise ValueError(f"Constraint missing key '{k}': {c}")

    return model

In [ ]:
# -----------------------------
# Optional fallback parser
# -----------------------------
# If LLM JSON parsing fails, this parser handles simple LP/MILP templates.
# Template example:
# "Maximize 3 x1 + 5 x2 subject to 2 x1 + x2 <= 100, x1 + 3 x2 <= 90, x1 >= 0, x2 >= 0"

term_pattern = re.compile(r"([+-]?\s*\d*\.?\d*)\s*\*?\s*([A-Za-z]\w*)")

def parse_linear_expr(expr: str) -> Dict[str, float]:
    expr = expr.replace("-", "+-")
    parts = [p.strip() for p in expr.split("+") if p.strip()]
    coeffs: Dict[str, float] = {}

    for p in parts:
        m = term_pattern.fullmatch(p)
        if not m:
            continue
        raw_coeff, var = m.groups()
        raw_coeff = raw_coeff.replace(" ", "")
        if raw_coeff in ["", "+"]:
            val = 1.0
        elif raw_coeff == "-":
            val = -1.0
        else:
            val = float(raw_coeff)
        coeffs[var] = coeffs.get(var, 0.0) + val
    return coeffs


def fallback_model_from_text(problem_text: str) -> Dict[str, Any]:
    text = " ".join(problem_text.strip().split())
    low = text.lower()

    if "maximize" in low:
        sense = "max"
        split_token = "maximize"
    elif "minimize" in low:
        sense = "min"
        split_token = "minimize"
    else:
        raise ValueError("Fallback parser: problem must include 'maximize' or 'minimize'.")

    if "subject to" not in low:
        raise ValueError("Fallback parser: problem must include 'subject to'.")

    # Case-insensitive split by using index on lower version.
    i_obj_start = low.find(split_token) + len(split_token)
    i_st = low.find("subject to")
    obj_expr = text[i_obj_start:i_st].strip(" :")
    constraints_block = text[i_st + len("subject to"):].strip()

    objective_terms = parse_linear_expr(obj_expr)
    if not objective_terms:
        raise ValueError("Fallback parser: could not parse objective expression.")

    raw_constraints = [c.strip() for c in re.split(r"[,;]", constraints_block) if c.strip()]
    constraints = []
    all_vars = set(objective_terms.keys())

    for idx, c in enumerate(raw_constraints, 1):
        m = re.match(r"(.+?)(<=|>=|=)(.+)", c)
        if not m:
            continue
        lhs_expr, sgn, rhs_txt = m.groups()
        lhs_terms = parse_linear_expr(lhs_expr.strip())
        rhs_val = float(rhs_txt.strip())
        all_vars.update(lhs_terms.keys())
        constraints.append({
            "name": f"c{idx}",
            "lhs": lhs_terms,
            "sense": sgn,
            "rhs": rhs_val,
        })

    if not constraints:
        raise ValueError("Fallback parser: no constraints parsed.")

    variables = {
        v: {"type": "continuous", "low": 0, "up": None}
        for v in sorted(all_vars)
    }

    # Lightweight integer detection, e.g., "x1, x2 are integers" or "x1 integer"
    int_hits = re.findall(r"([A-Za-z]\w*)\s+integer", low)
    for v in int_hits:
        if v in variables:
            variables[v]["type"] = "integer"

    bin_hits = re.findall(r"([A-Za-z]\w*)\s+binary", low)
    for v in bin_hits:
        if v in variables:
            variables[v]["type"] = "binary"
            variables[v]["low"] = 0
            variables[v]["up"] = 1

    return {
        "problem_type": "MILP" if any(variables[v]["type"] != "continuous" for v in variables) else "LP",
        "objective": {"sense": sense, "terms": objective_terms},
        "variables": variables,
        "constraints": constraints,
    }

In [ ]:
# -----------------------------
# Prompt templates
# -----------------------------

FORMULATOR_PROMPT = """
You are an optimization model formulator.
Convert the natural-language optimization problem into STRICT JSON.
Return ONLY JSON (no markdown, no explanation).

JSON schema:
{
  "problem_type": "LP" or "MILP",
  "objective": {
    "sense": "max" or "min",
    "terms": {"x1": 3, "x2": 5}
  },
  "variables": {
    "x1": {"type": "continuous|integer|binary", "low": 0, "up": null},
    "x2": {"type": "continuous|integer|binary", "low": 0, "up": null}
  },
  "constraints": [
    {"name": "c1", "lhs": {"x1": 2, "x2": 1}, "sense": "<=", "rhs": 100}
  ]
}

Rules:
- All expressions must be linear.
- Use numeric coefficients.
- If bounds are not stated, use low=0 and up=null.
- If variable type is not stated, use continuous.

Problem:
{problem}
""".strip()

CRITIC_PROMPT = """
You are an optimization model critic.
Given the original problem and a candidate JSON model, fix any mistakes.
Return ONLY corrected JSON following the same schema.

Original problem:
{problem}

Candidate JSON:
{candidate_json}
""".strip()

ANALYSIS_PROMPT = """
You are an operations research analyst.
Write a concise analysis of the solve result.
If status is Optimal, explain what the numbers mean.
If not solvable (Infeasible, Unbounded, etc.), explain likely causes and next checks.

Problem:
{problem}

Model JSON:
{model_json}

Solve Summary:
{summary}
""".strip()

In [ ]:
# -----------------------------
# Solver and orchestration
# -----------------------------

def build_and_solve(model: Dict[str, Any]) -> Dict[str, Any]:
    model = normalize_schema(model)

    sense = model["objective"]["sense"].lower()
    prob = pulp.LpProblem("multiagent_optimization", pulp.LpMaximize if sense == "max" else pulp.LpMinimize)

    var_map = {}
    for v, meta in model["variables"].items():
        vtype = meta.get("type", "continuous").lower()
        if vtype == "binary":
            cat = pulp.LpBinary
            low, up = 0, 1
        elif vtype == "integer":
            cat = pulp.LpInteger
            low, up = meta.get("low", 0), meta.get("up", None)
        else:
            cat = pulp.LpContinuous
            low, up = meta.get("low", 0), meta.get("up", None)

        var_map[v] = pulp.LpVariable(v, lowBound=low, upBound=up, cat=cat)

    # Objective
    obj_terms = model["objective"]["terms"]
    prob += pulp.lpSum(float(c) * var_map[v] for v, c in obj_terms.items())

    # Constraints
    for c in model["constraints"]:
        lhs = pulp.lpSum(float(coeff) * var_map[v] for v, coeff in c["lhs"].items())
        sgn = c["sense"]
        rhs = float(c["rhs"])
        name = c.get("name", "")

        if sgn == "<=":
            prob += (lhs <= rhs), name
        elif sgn == ">=":
            prob += (lhs >= rhs), name
        elif sgn == "=":
            prob += (lhs == rhs), name
        else:
            raise ValueError(f"Invalid constraint sense: {sgn}")

    # Solve with CBC (bundled with PuLP in Colab in most cases)
    solver = pulp.PULP_CBC_CMD(msg=False)
    prob.solve(solver)

    status = pulp.LpStatus[prob.status]
    result = {
        "status": status,
        "objective_value": None,
        "variables": {},
    }

    if status == "Optimal":
        result["objective_value"] = float(pulp.value(prob.objective))
        result["variables"] = {v: float(var_map[v].value()) for v in var_map}

    return result


def formulate_with_agents(problem_text: str) -> Tuple[Dict[str, Any], Dict[str, str]]:
    logs = {}

    # Agent 1: formulate initial JSON.
    p1 = FORMULATOR_PROMPT.format(problem=problem_text)
    raw1 = formulator_agent.run(p1)
    logs["formulator_raw"] = raw1

    try:
        candidate = parse_json_output(raw1)
    except Exception:
        candidate = fallback_model_from_text(problem_text)
        logs["formulator_parse"] = "Failed -> fallback parser used."

    # Agent 2: critique and correct.
    p2 = CRITIC_PROMPT.format(problem=problem_text, candidate_json=json.dumps(candidate, indent=2))
    raw2 = critic_agent.run(p2)
    logs["critic_raw"] = raw2

    try:
        corrected = parse_json_output(raw2)
    except Exception:
        corrected = candidate
        logs["critic_parse"] = "Failed -> kept candidate model."

    corrected = normalize_schema(corrected)
    return corrected, logs


def run_multiagent_optimization(problem_text: str, show_debug: bool = False) -> Dict[str, Any]:
    model, logs = formulate_with_agents(problem_text)
    solve_result = build_and_solve(model)

    summary = {
        "status": solve_result["status"],
        "objective_value": solve_result["objective_value"],
        "variables": solve_result["variables"],
    }

    # Use critic agent again as analyst to explain result in natural language.
    analysis_prompt = ANALYSIS_PROMPT.format(
        problem=problem_text,
        model_json=json.dumps(model, indent=2),
        summary=json.dumps(summary, indent=2),
    )
    analysis_text = critic_agent.run(analysis_prompt)

    out = {
        "model": model,
        "solve_result": solve_result,
        "analysis": analysis_text,
        "logs": logs,
    }

    if show_debug:
        display(Markdown("### Debug Logs"))
        print(json.dumps(logs, indent=2)[:4000])

    return out

In [ ]:
# -----------------------------
# Example 1: Feasible MILP
# -----------------------------

problem_1 = """
Maximize profit = 40 x1 + 30 x2
subject to
2 x1 + x2 <= 40,
x1 + 2 x2 <= 50,
x1 <= 20,
x1 >= 0,
x2 >= 0,
x1 integer,
x2 integer
"""

res1 = run_multiagent_optimization(problem_1, show_debug=False)

print("Status:", res1["solve_result"]["status"])
print("Objective:", res1["solve_result"]["objective_value"])
print("Variables:", res1["solve_result"]["variables"])

display(Markdown("### Analysis"))
print(res1["analysis"])

In [ ]:
# -----------------------------
# Example 2: Infeasible LP
# -----------------------------

problem_2 = """
Minimize cost = 3 x + 2 y
subject to
x + y >= 10,
x + y <= 5,
x >= 0,
y >= 0
"""

res2 = run_multiagent_optimization(problem_2, show_debug=False)

print("Status:", res2["solve_result"]["status"])
print("Objective:", res2["solve_result"]["objective_value"])
print("Variables:", res2["solve_result"]["variables"])

display(Markdown("### Analysis"))
print(res2["analysis"])

## Notes

- This framework is intentionally transparent: each stage (formulation, critique, solve, analysis) is inspectable.
- If a model output is malformed JSON, a lightweight fallback parser tries simple LP/MILP text formats.
- For stronger production performance, you can replace the local Hugging Face models with larger instruct models or an API-based model while keeping the same agent architecture.
